# TRI-X-CDSS Advanced Case Comparison

This notebook compares multiple dizziness presentations across Triage, SRGL, DRAS-5, and ORASR, then visualizes how urgency and pathway decisions change by case profile.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from trix_cdss.core.triage import DizzinessSymptoms, PatientRiskFactors, VitalSigns, perform_dizziness_triage
from trix_cdss.core.srgl import perform_red_flag_screening
from trix_cdss.core.dras5 import DRAS5Features, classify_urgency_level
from trix_cdss.core.orasr import PatientContext, ResourceAvailability, route_to_care_pathway
from trix_cdss.core.titrate import ClinicalFindings, TimePoint, perform_time_bounded_assessment

## Define comparison cases

In [ ]:
cases = {
    'Likely BPPV': {
        'vitals': VitalSigns(78, 132, 82, 16, 36.7, 98),
        'symptoms': DizzinessSymptoms('spinning', 12, 30, positional_trigger=True, continuous_vertigo=False),
        'risk': PatientRiskFactors(age=58, hypertension=True),
        'srgl': {'symptoms': {'headache': False}, 'vital_signs': {'heart_rate': 78, 'systolic_bp': 132}, 'neurological_exam': {'diplopia': False, 'ataxia': False, 'dysarthria': False}, 'medical_history': {'anticoagulation': False}},
        'features': DRAS5Features(age=58, gender='female', hypertension=True, symptom_duration_hours=12, symptom_severity=4, positional_trigger=True, continuous_vertigo=False, stroke_probability=0.08),
        'clinical_findings': ClinicalFindings(time_point=TimePoint.T0, timestamp=datetime.now(), heart_rate=78, blood_pressure_sys=132, blood_pressure_dia=82, hints_performed=True, hints_hit_result='positive', hints_nystagmus='horizontal', hints_skew=False, symptom_severity=4),
    },
    'Possible Stroke': {
        'vitals': VitalSigns(96, 178, 102, 20, 36.9, 97),
        'symptoms': DizzinessSymptoms('continuous vertigo', 1.5, None, diplopia=True, ataxia=True, continuous_vertigo=True),
        'risk': PatientRiskFactors(age=74, atrial_fibrillation=True, hypertension=True, prior_stroke_tia=True),
        'srgl': {'symptoms': {'headache': False, 'diplopia': True, 'ataxia': True}, 'vital_signs': {'heart_rate': 96, 'systolic_bp': 178}, 'neurological_exam': {'diplopia': True, 'ataxia': True, 'dysarthria': False}, 'medical_history': {'anticoagulation': True}},
        'features': DRAS5Features(age=74, gender='male', atrial_fibrillation=True, hypertension=True, prior_stroke=True, symptom_duration_hours=1.5, symptom_severity=8, hints_performed=True, hints_central=True, ataxia=True, diplopia=True, systolic_bp=178, heart_rate=96, stroke_probability=0.9, continuous_vertigo=True),
        'clinical_findings': ClinicalFindings(time_point=TimePoint.T0, timestamp=datetime.now(), heart_rate=96, blood_pressure_sys=178, blood_pressure_dia=102, hints_performed=True, hints_hit_result='negative', hints_nystagmus='direction_changing', hints_skew=True, ataxia=True, symptom_severity=8),
    },
    'Intermediate Vestibular': {
        'vitals': VitalSigns(88, 146, 90, 18, 36.8, 98),
        'symptoms': DizzinessSymptoms('vertigo', 6, None, nausea_vomiting=True, continuous_vertigo=True),
        'risk': PatientRiskFactors(age=63, hypertension=True, diabetes=True),
        'srgl': {'symptoms': {'headache': False}, 'vital_signs': {'heart_rate': 88, 'systolic_bp': 146}, 'neurological_exam': {'diplopia': False, 'ataxia': False, 'dysarthria': False}, 'medical_history': {'anticoagulation': False}},
        'features': DRAS5Features(age=63, gender='female', hypertension=True, diabetes=True, symptom_duration_hours=6, symptom_severity=6, hints_performed=True, hints_central=False, systolic_bp=146, heart_rate=88, stroke_probability=0.32, continuous_vertigo=True),
        'clinical_findings': ClinicalFindings(time_point=TimePoint.T0, timestamp=datetime.now(), heart_rate=88, blood_pressure_sys=146, blood_pressure_dia=90, hints_performed=True, hints_hit_result='positive', hints_nystagmus='horizontal', hints_skew=False, symptom_severity=6),
    },
}

## Run the full comparison workflow

In [ ]:
results = []
for case_name, case in cases.items():
    triage = perform_dizziness_triage(case['vitals'], case['symptoms'], case['risk'])
    screening = perform_red_flag_screening(**case['srgl'])
    titrate = perform_time_bounded_assessment(
        patient_id=case_name.replace(' ', '_'),
        initial_presentation_time=datetime.now(),
        time_point=TimePoint.T0,
        findings=case['clinical_findings'],
    )
    dras = classify_urgency_level(
        case['features'],
        esi_level=triage.esi_level.value,
        red_flags_critical=screening.has_critical_red_flags(),
        titrate_risk_score=8.5 if case_name == 'Possible Stroke' else 4.5 if case_name == 'Intermediate Vestibular' else 1.5,
    )
    routing = route_to_care_pathway(
        dras_level=dras.level.value,
        patient_context=PatientContext(can_travel_independently=case_name != 'Possible Stroke', has_transportation=True, home_support=True),
        resource_availability=ResourceAvailability(),
        clinical_features={'stroke_probability': case['features'].stroke_probability, 'hints_central': case['features'].hints_central, 'severe': case['features'].symptom_severity >= 8},
        red_flags_present=screening.has_critical_red_flags(),
    )
    results.append({
        'case': case_name,
        'esi': triage.esi_level.value,
        'red_flags': screening.has_critical_red_flags(),
        'dras': dras.level.value,
        'pathway': routing.primary_pathway.value,
        'stroke_probability': case['features'].stroke_probability,
        'titrate_summary': titrate.get_assessment_summary(),
    })

results

## Visual comparison

In [ ]:
labels = [row['case'] for row in results]
esi_levels = [row['esi'] for row in results]
dras_levels = [row['dras'] for row in results]
stroke_probs = [row['stroke_probability'] for row in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = range(len(labels))

axes[0].bar(x, esi_levels, width=0.35, label='ESI', color='#457b9d')
axes[0].bar([i + 0.35 for i in x], dras_levels, width=0.35, label='DRAS-5', color='#d62828')
axes[0].set_xticks([i + 0.175 for i in x], labels, rotation=20, ha='right')
axes[0].set_ylim(0, 5.5)
axes[0].set_title('Urgency comparison by case')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(labels, stroke_probs, color=['#2a9d8f', '#d62828', '#e9c46a'], edgecolor='black')
axes[1].set_ylim(0, 1.0)
axes[1].set_title('Stroke probability by case')
axes[1].set_ylabel('Probability')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(True, axis='y', alpha=0.3)

fig.tight_layout()

## Routing summary

In [ ]:
[{key: row[key] for key in ['case', 'esi', 'red_flags', 'dras', 'pathway']} for row in results]